# LECTURE 14 — DEMONSTRATION
### MODULE 6: NEURAL NETWORKS & DEEP LEARNING

**DEMONSTRATION** — Feedforward Neural Networks for Inflation Forecasting

*WAIFEM ML Facilitation Programme 2026 — Compatible with Google Colab & VS Code*

### Package Installation

Uncomment and run the cell below if any package is missing:

In [ ]:
!pip install tensorflow scikit-learn matplotlib pandas numpy seaborn statsmodels scipy

## ── SECTION 1: IMPORTS & ENVIRONMENT SETUP

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    import seaborn as sns
except ImportError as exc:
    raise ImportError(
        "Missing required package 'seaborn'. Install it with:\n"
        "python -m pip install seaborn\n"
        "or in a notebook: !pip install seaborn"
    ) from exc

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'   # suppress TensorFlow info messages

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 11, 'axes.labelsize': 10})

# Detect execution environment
try:
    import google.colab          # noqa: F401
    IN_COLAB = True
    print("Environment : Google Colab")
except ImportError:
    IN_COLAB = False
    print("Environment : Local (VS Code / terminal)")

print(f"TensorFlow  : {tf.__version__}")
print(f"NumPy       : {np.__version__}")

## ── SECTION 2: SYNTHETIC DATA GENERATION

In [ ]:
def generate_macro_data(n_obs: int = 240, seed: int = 42) -> pd.DataFrame:
    """
    Generate 240 monthly observations (20 years) of synthetic macroeconomic data.

    Features (X) : gdp_growth, money_supply_growth, exchange_rate_chg,
                   oil_price_chg, interest_rate, fiscal_deficit
    Target  (y)  : inflation
    """
    rng   = np.random.default_rng(seed)
    dates = pd.date_range(start='2004-01-01', periods=n_obs, freq='MS')
    t     = np.linspace(0, 1, n_obs)

    gdp_growth          = 3.5  + 1.5 * np.sin(8  * np.pi * t) + rng.normal(0, 0.40, n_obs)
    money_supply_growth = 10.0 + np.cumsum(rng.normal(0, 0.12, n_obs)) + rng.normal(0, 0.80, n_obs)
    exchange_rate_chg   = rng.normal(0, 1.8, n_obs)
    oil_price_chg       = 9.0  * np.sin(6  * np.pi * t) + rng.normal(0, 4.00, n_obs)
    interest_rate       = 8.0  + 2.0 * np.sin(4  * np.pi * t) + rng.normal(0, 0.50, n_obs)
    fiscal_deficit      = -3.0 + 1.5 * np.sin(3  * np.pi * t) + rng.normal(0, 0.40, n_obs)

    inflation = (
        4.5
        + 0.28 * money_supply_growth
        + 0.14 * oil_price_chg
        + 0.18 * exchange_rate_chg
        - 0.22 * gdp_growth
        - 0.10 * interest_rate
        + 0.09 * fiscal_deficit
        + rng.normal(0, 0.70, n_obs)
    )

    return pd.DataFrame({
        'inflation':            inflation,
        'gdp_growth':           gdp_growth,
        'money_supply_growth':  money_supply_growth,
        'exchange_rate_chg':    exchange_rate_chg,
        'oil_price_chg':        oil_price_chg,
        'interest_rate':        interest_rate,
        'fiscal_deficit':       fiscal_deficit,
    }, index=dates)


df = generate_macro_data()
print(f"\nDataset : {df.shape[0]} monthly observations  x  {df.shape[1]} variables")
print(df.describe().round(2).to_string())

## ── SECTION 3: EXPLORATORY VISUALISATION

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(14, 11))
fig.suptitle('Synthetic Macroeconomic Variables — Monthly (2004–2024)',
             fontsize=13, fontweight='bold', y=1.01)

palette = sns.color_palette('tab10', len(df.columns))
for ax, col, c in zip(axes.flatten(), df.columns, palette):
    ax.plot(df.index, df[col], color=c, linewidth=1.2)
    ax.set_title(col.replace('_', ' ').title())
    ax.tick_params(axis='x', rotation=30)
    ax.grid(True, alpha=0.3)

axes.flatten()[-1].set_visible(False)      # hide the spare 8th panel
plt.tight_layout()
plt.savefig('lecture14_demo_raw_data.png', bbox_inches='tight')
plt.show()
print("Saved : lecture14_demo_raw_data.png")

# Correlation heatmap
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, square=True, ax=ax)
ax.set_title('Correlation Matrix — Macroeconomic Variables', fontweight='bold')
plt.tight_layout()
plt.savefig('lecture14_demo_correlation.png', bbox_inches='tight')
plt.show()
print("Saved : lecture14_demo_correlation.png")

## ── SECTION 4: PREPROCESSING

In [ ]:
FEATURE_COLS = ['gdp_growth', 'money_supply_growth', 'exchange_rate_chg',
                'oil_price_chg', 'interest_rate', 'fiscal_deficit']
TARGET_COL   = 'inflation'

X = df[FEATURE_COLS].values
y = df[TARGET_COL].values.reshape(-1, 1)

scaler_X = MinMaxScaler(feature_range=(0, 1))
scaler_y = MinMaxScaler(feature_range=(0, 1))
X_sc     = scaler_X.fit_transform(X)
y_sc     = scaler_y.fit_transform(y).ravel()

# Chronological 80/20 split — NO shuffle (preserves time order)
SPLIT       = int(0.80 * len(X_sc))
X_train, X_test = X_sc[:SPLIT], X_sc[SPLIT:]
y_train, y_test = y_sc[:SPLIT], y_sc[SPLIT:]

print(f"\nTrain : {X_train.shape[0]} observations  |  Test : {X_test.shape[0]} observations")

## ── SECTION 5: MODEL ARCHITECTURE

In [ ]:
def build_ffnn(n_features: int,
               layer_sizes: tuple = (128, 64, 32),
               dropout_rate: float = 0.25,
               learning_rate: float = 5e-4) -> tf.keras.Model:
    """Feedforward neural network for single-step regression."""
    model = Sequential(name='Inflation_FFNN')
    for i, units in enumerate(layer_sizes):
        input_kw = {'input_shape': (n_features,)} if i == 0 else {}
        model.add(Dense(units, activation='relu', name=f'dense_{i+1}', **input_kw))
        model.add(BatchNormalization(name=f'bn_{i+1}'))
        model.add(Dropout(dropout_rate, name=f'drop_{i+1}'))
    model.add(Dense(1, activation='linear', name='output'))
    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss='mse',
        metrics=['mae'],
    )
    return model


model = build_ffnn(n_features=X_train.shape[1])
model.summary()

## ── SECTION 6: TRAINING

In [ ]:
callbacks = [
    EarlyStopping(
        monitor='val_loss', patience=25,
        restore_best_weights=True, verbose=0,
    ),
    ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=10, min_lr=1e-6, verbose=0,
    ),
]

history = model.fit(
    X_train, y_train,
    validation_split=0.15,
    epochs=300,
    batch_size=16,
    callbacks=callbacks,
    verbose=0,
)

best_epoch = int(np.argmin(history.history['val_loss'])) + 1
print(f"\nTraining complete — best epoch : {best_epoch} / {len(history.history['loss'])}")

## ── SECTION 7: LEARNING CURVES (Overfitting Diagnosis)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Learning Curves — Feedforward Neural Network (Inflation Model)',
             fontsize=12, fontweight='bold')

for ax, (tr_key, val_key), title in zip(
    axes,
    [('loss', 'val_loss'), ('mae', 'val_mae')],
    ['Loss (MSE)', 'Mean Absolute Error (MAE)'],
):
    ax.plot(history.history[tr_key],  label='Train',      color='steelblue', linewidth=1.5)
    ax.plot(history.history[val_key], label='Validation', color='tomato',    linewidth=1.5, linestyle='--')
    ax.axvline(best_epoch - 1, color='grey', linestyle=':', linewidth=1.2,
               label=f'Best epoch ({best_epoch})')
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('lecture14_demo_learning_curves.png', bbox_inches='tight')
plt.show()
print("Saved : lecture14_demo_learning_curves.png")

print("\n── Overfitting Diagnostics Guide ────────────────────────────────────")
print("  Train loss DOWN + Val loss DOWN together  →  Healthy learning")
print("  Train loss DOWN + Val loss UP (diverging) →  Overfitting")
print("     Remedy: increase Dropout, reduce layer sizes, add L2 regularisation")
print("  Both losses remain HIGH                   →  Underfitting")
print("     Remedy: increase capacity, reduce Dropout, train longer")

## ── SECTION 8: TEST-SET EVALUATION

In [ ]:
y_pred_sc = model.predict(X_test, verbose=0).ravel()
y_pred    = scaler_y.inverse_transform(y_pred_sc.reshape(-1, 1)).ravel()
y_actual  = scaler_y.inverse_transform(y_test.reshape(-1, 1)).ravel()

rmse = np.sqrt(mean_squared_error(y_actual, y_pred))
mae  = mean_absolute_error(y_actual, y_pred)
r2   = r2_score(y_actual, y_pred)

print(f"\n{'─'*44}")
print(f"  {'Metric':<22} {'Value':>10}")
print(f"{'─'*44}")
print(f"  {'RMSE (percentage points)':<22} {rmse:>10.4f}")
print(f"  {'MAE  (percentage points)':<22} {mae:>10.4f}")
print(f"  {'R-squared (R²)':<22} {r2:>10.4f}")
print(f"{'─'*44}")

## ── SECTION 9: PREDICTIONS PLOT

In [ ]:
test_dates = df.index[SPLIT:]

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(test_dates, y_actual, label='Actual Inflation',  color='steelblue', linewidth=2.0)
ax.plot(test_dates, y_pred,   label='FFNN Forecast',     color='tomato',    linewidth=2.0, linestyle='--')
ax.fill_between(test_dates, y_actual, y_pred, alpha=0.12, color='grey', label='Forecast Error')
ax.set_title(
    f'Feedforward NN — Out-of-Sample Inflation Forecast  '
    f'|  RMSE = {rmse:.3f} pp  |  R² = {r2:.3f}',
    fontsize=11, fontweight='bold',
)
ax.set_xlabel('Date')
ax.set_ylabel('Inflation Rate (%)')
ax.legend()
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.savefig('lecture14_demo_predictions.png', bbox_inches='tight')
plt.show()
print("Saved : lecture14_demo_predictions.png")

## ── SECTION 10: SUMMARY

In [ ]:
print("""
╔═══════════════════════════════════════════════════════════════════╗
║  LECTURE 14 DEMO COMPLETE                                                                        ║
╠═══════════════════════════════════════════════════════════════════╣
║  Concepts demonstrated:                                                                                    ║
║   1. Synthetic macroeconomic time-series generation                                         ║
║   2. MinMaxScaler preprocessing for neural networks                                        ║
║   3. Feedforward NN — Dense / BatchNorm / Dropout blocks                            ║
║   4. EarlyStopping + ReduceLROnPlateau training callbacks                                 ║
║   5. Learning curve interpretation (overfitting diagnosis)                                   ║
║   6. RMSE | MAE | R² evaluation on a held-out test set                                     ║
╠═══════════════════════════════════════════════════════════════════╣
║  Saved outputs:                                                                                                  ║
║   lecture14_demo_raw_data.png                                                                        ║
║   lecture14_demo_correlation.png                                                                     ║
║   lecture14_demo_learning_curves.png                                                              ║
║   lecture14_demo_predictions.png                                                                     ║
╚═══════════════════════════════════════════════════════════════════╝
""")

## ── SECTION 11: MODEL COMPARISON — ECONOMETRIC BENCHMARKS vs DEEP LEARNING

The FFNN is benchmarked against **six traditional econometric models** across three families,
all evaluated on the **same 20% hold-out test set** (48 monthly observations).

| Family | Models |
|---|---|
| Univariate AR | Random Walk · AR(p) · TVP-AR · TVP-AR-SV |
| Multivariate VAR | VAR(p) · TVP-VAR · TVP-VAR-SV |
| Deep Learning | FFNN (Sections 5–8) |

**Evaluation metrics** — RMSE, MAE, R²  
**Statistical test** — Diebold-Mariano (1995) test for equal predictive accuracy (Harvey et al. 1997 correction)

> **Note on SV models:** Stochastic volatility captures *time-varying uncertainty*, not a different conditional mean.  
> For point forecast metrics (RMSE, MAE): TVP-AR-SV ≈ TVP-AR and TVP-VAR-SV ≈ TVP-VAR.  
> The SV value-add is visualised as time-varying prediction intervals in the chart below.

**TVP implementations:**
- *TVP-AR / TVP-AR-SV* — exact Kalman filter; AR coefficients follow independent random walks
- *TVP-VAR / TVP-VAR-SV* — rolling-window VAR (60-month window); SV approximated via EWMA (λ = 0.94)

In [ ]:
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.vector_ar.var_model import VAR
from scipy import stats

# ── Metric helper ─────────────────────────────────────────────────────────
def eval_metrics(actual, predicted, name):
    return {'Model': name,
            'RMSE':  float(np.sqrt(mean_squared_error(actual, predicted))),
            'MAE':   float(mean_absolute_error(actual, predicted)),
            'R²':    float(r2_score(actual, predicted))}

# ── Diebold-Mariano test (Harvey et al. 1997 small-sample correction) ─────
def diebold_mariano(e1, e2, h=1):
    """
    H₀: equal MSE-loss predictive accuracy.
    Positive DM → e1 has larger squared errors → e2 is more accurate.
    Uses Newey-West long-run variance with Harvey et al. t-correction.
    """
    d    = e1**2 - e2**2
    n    = len(d);  d_bar = d.mean();  dc = d - d_bar
    lrv  = float(np.dot(dc, dc)) / n
    for k in range(1, h):
        lrv += 2 * (1 - k / h) * float(np.dot(dc[k:], dc[:-k])) / n
    if lrv <= 0:
        return np.nan, np.nan
    dm = (d_bar / np.sqrt(lrv / n)) * np.sqrt((n + 1 - 2*h + h*(h-1)/n) / n)
    return float(dm), float(2 * stats.t.sf(abs(dm), df=n - 1))

print("Helpers loaded: eval_metrics, diebold_mariano")

In [ ]:
# ── Raw (unscaled) data for traditional econometric models ────────────────
y_full   = df[TARGET_COL].values          # all 240 inflation observations
n_test   = len(y_full) - SPLIT
test_idx = df.index[SPLIT:]

# VAR uses a 4-variable system (keeps model parsimonious; ~36 params on 190 obs)
VAR_COLS = ['inflation', 'gdp_growth', 'money_supply_growth', 'interest_rate']
data_var = df[VAR_COLS].values            # shape (240, 4); inflation is column 0

print(f"Test window : {test_idx[0].strftime('%Y-%m')} → {test_idx[-1].strftime('%Y-%m')}  ({n_test} obs)")
print(f"VAR cols    : {VAR_COLS}")

# ── 11.1  Random Walk (ŷ_t = y_{t-1}) — hard-to-beat naive benchmark ─────
rw_preds = y_full[SPLIT - 1: SPLIT + n_test - 1]
print(f"\nRandom Walk  RMSE = {np.sqrt(mean_squared_error(y_full[SPLIT:], rw_preds)):.4f}")

In [ ]:
# ── 11.2  AR(p) — AIC lag selection, expanding-window 1-step-ahead ───────
aic_ar = {}
for lag in range(1, 13):
    try:
        aic_ar[lag] = AutoReg(y_full[:SPLIT], lags=lag, old_names=False).fit().aic
    except Exception:
        pass
ar_lag = min(aic_ar, key=aic_ar.get)
print(f"AR — AIC-optimal lag: {ar_lag}  (AIC = {aic_ar[ar_lag]:.2f})")

ar_preds = []
for t in range(n_test):
    m = AutoReg(y_full[:SPLIT + t], lags=ar_lag, old_names=False).fit()
    ar_preds.append(float(m.forecast(steps=1)))
ar_preds = np.array(ar_preds)
print(f"AR({ar_lag})  RMSE = {np.sqrt(mean_squared_error(y_full[SPLIT:], ar_preds)):.4f}")

In [ ]:
# ── 11.3  VAR(p) — AIC lag selection (capped 1-4), expanding-window ──────
aic_var = {}
for lag in range(1, 5):
    try:
        aic_var[lag] = VAR(data_var[:SPLIT]).fit(maxlags=lag, ic=None).aic
    except Exception:
        pass
var_lag = min(aic_var, key=aic_var.get)
print(f"VAR — AIC-optimal lag: {var_lag}")

var_preds = []
for t in range(n_test):
    sub = data_var[:SPLIT + t]
    try:
        m  = VAR(sub).fit(maxlags=var_lag, ic=None)
        fc = m.forecast(sub[-m.k_ar:], steps=1)
        var_preds.append(float(fc[0, 0]))
    except Exception:
        var_preds.append(np.nan)
var_preds = np.array(var_preds)
print(f"VAR({var_lag})  RMSE = {np.sqrt(mean_squared_error(y_full[SPLIT:], var_preds)):.4f}")

In [ ]:
# ── 11.4  TVP-AR (Kalman Filter) ─────────────────────────────────────────
# State:  β_t = [intercept, β_1, ..., β_p]  (each coefficient follows a random walk)
# Obs:    y_t  = x_t' β_t + ε_t,   ε_t ~ N(0, R)
# Trans:  β_t  = β_{t-1} + η_t,     η_t ~ N(0, Q)
#
# ── 11.5  TVP-AR-SV (same Kalman filter + EWMA time-varying R) ───────────
# Setting sv_lambda updates R via EWMA after each observation.

def _tvp_ar_core(y, train_end, p=2, Q_scale=1e-4, sv_lambda=None):
    """
    Kalman filter for TVP-AR(p).
    sv_lambda activates EWMA stochastic volatility on the observation variance R.
    Returns (forecasts array, sigma2_path array).
    """
    ns   = p + 1
    y_tr = y[:train_end]

    # OLS initialisation on training data
    X0   = np.column_stack([np.ones(train_end - p)] +
                            [y_tr[p-i-1:train_end-i-1] for i in range(p)])
    b0   = np.linalg.lstsq(X0, y_tr[p:], rcond=None)[0]
    R    = float(np.var(y_tr[p:] - X0 @ b0))
    Q    = np.eye(ns) * Q_scale * R
    P    = np.eye(ns) * R
    beta = b0.copy()

    def _kf_step(beta, P, x, y_obs):
        b_pr = beta;  P_pr = P + Q
        v    = float(y_obs - x @ b_pr)
        F    = float(x @ P_pr @ x) + R
        K    = P_pr @ x / F
        return b_pr + K * v, P_pr - np.outer(K, x) @ P_pr, v

    # Filter through training period (update state only, no forecasts recorded)
    for t in range(p, train_end):
        xt         = np.array([1.] + [y[t-i-1] for i in range(p)])
        beta, P, v = _kf_step(beta, P, xt, y[t])
        if sv_lambda is not None:
            R = sv_lambda * R + (1 - sv_lambda) * v**2

    # Recursive 1-step-ahead forecasts on test period
    preds, sig2 = [], []
    for t in range(train_end, len(y)):
        xt = np.array([1.] + [y[t-i-1] for i in range(p)])
        preds.append(float(xt @ beta))
        sig2.append(R)
        beta, P, v = _kf_step(beta, P, xt, y[t])
        if sv_lambda is not None:
            R = sv_lambda * R + (1 - sv_lambda) * v**2

    return np.array(preds), np.array(sig2)


tvpar_preds,    _              = _tvp_ar_core(y_full, SPLIT, p=2)
tvpar_sv_preds, tvpar_sv_sig2  = _tvp_ar_core(y_full, SPLIT, p=2, sv_lambda=0.94)

print(f"TVP-AR(2)     RMSE = {np.sqrt(mean_squared_error(y_full[SPLIT:], tvpar_preds)):.4f}")
print(f"TVP-AR-SV(2)  RMSE = {np.sqrt(mean_squared_error(y_full[SPLIT:], tvpar_sv_preds)):.4f}")
print("(TVP-AR-SV ≈ TVP-AR for point forecasts; SV value-add is in prediction intervals.)")

In [ ]:
# ── 11.6  TVP-VAR (Rolling-Window VAR) ──────────────────────────────────
# Time-varying parameters approximated by re-estimating VAR on a rolling window.
ROLL_WIN = 60    # 5-year rolling window (months)
TVP_P    = 2     # VAR lag order

tvpvar_preds = []
for t in range(SPLIT, SPLIT + n_test):
    sub = data_var[max(0, t - ROLL_WIN):t]
    try:
        m  = VAR(sub).fit(maxlags=TVP_P, ic=None)
        fc = m.forecast(sub[-m.k_ar:], steps=1)
        tvpvar_preds.append(float(fc[0, 0]))
    except Exception:
        tvpvar_preds.append(np.nan)
tvpvar_preds = np.array(tvpvar_preds)
print(f"TVP-VAR({TVP_P}, win={ROLL_WIN})  RMSE = {np.sqrt(mean_squared_error(y_full[SPLIT:], tvpvar_preds)):.4f}")

# ── 11.7  TVP-VAR-SV (Rolling Window + EWMA time-varying variance) ───────
# 1-step-ahead *point* forecast is identical to TVP-VAR.
# SV captures time-varying prediction interval width only.
LAM_VAR = 0.94
_m0     = VAR(data_var[:SPLIT]).fit(maxlags=TVP_P, ic=None)
_R_sv   = float(np.var(_m0.resid[:, 0]))

tvpvar_sv_preds, tvpvar_sv_sig2 = [], []
for t in range(SPLIT, SPLIT + n_test):
    sub = data_var[max(0, t - ROLL_WIN):t]
    tvpvar_sv_sig2.append(_R_sv)
    try:
        m  = VAR(sub).fit(maxlags=TVP_P, ic=None)
        fc = m.forecast(sub[-m.k_ar:], steps=1)
        pt = float(fc[0, 0])
    except Exception:
        pt = np.nan
    tvpvar_sv_preds.append(pt)
    if not np.isnan(pt):
        _R_sv = LAM_VAR * _R_sv + (1 - LAM_VAR) * (y_full[t] - pt) ** 2

tvpvar_sv_preds = np.array(tvpvar_sv_preds)
tvpvar_sv_sig2  = np.array(tvpvar_sv_sig2)
print(f"TVP-VAR-SV({TVP_P}, win={ROLL_WIN})  RMSE = {np.sqrt(mean_squared_error(y_full[SPLIT:], tvpvar_sv_preds)):.4f}")
print("(Point forecast = TVP-VAR; SV adds time-varying prediction intervals.)")

### ── 11.8  Results — Forecast Comparison Table & Diebold-Mariano Test

In [ ]:
y_test_raw = y_full[SPLIT:]    # unscaled actuals — identical to y_actual from Section 8

model_preds_dict = {
    'Random Walk':  rw_preds,
    'AR(p)':        ar_preds,
    'VAR(p)':       var_preds,
    'TVP-AR':       tvpar_preds,
    'TVP-AR-SV':    tvpar_sv_preds,
    'TVP-VAR':      tvpvar_preds,
    'TVP-VAR-SV':   tvpvar_sv_preds,
    'FFNN (DNN)':   y_pred,         # from Section 8
}

# ── Comparison table ──────────────────────────────────────────────────────
results_df = (pd.DataFrame([eval_metrics(y_test_raw, v, k)
                             for k, v in model_preds_dict.items()])
                .sort_values('RMSE').reset_index(drop=True))
results_df.index += 1

print('\n' + '═'*62)
print('  OUT-OF-SAMPLE FORECAST COMPARISON — Inflation (pp)')
print('═'*62)
print(results_df.to_string(float_format='{:.4f}'.format))
print('═'*62)
print(f"\n  Best model  : {results_df.iloc[0]['Model']}")
rw_rmse = results_df[results_df['Model'] == 'Random Walk']['RMSE'].values[0]
print(f"  RW baseline : RMSE = {rw_rmse:.4f}")

# ── Diebold-Mariano tests — each rival vs FFNN ────────────────────────────
e_ffnn = y_test_raw - y_pred

print('\n' + '─'*70)
print("  DIEBOLD-MARIANO TEST  — H₀: equal forecast accuracy vs FFNN (DNN)")
print("  DM > 0  →  FFNN has lower MSE (FFNN beats rival)")
print("  DM < 0  →  Rival has lower MSE (rival beats FFNN)")
print('─'*70)
print(f"  {'Model':<22} {'DM stat':>9} {'p-value':>9}  {'':>4}")
print('─'*70)
for name, preds in model_preds_dict.items():
    if name == 'FFNN (DNN)':
        continue
    dm, pv = diebold_mariano(y_test_raw - preds, e_ffnn)
    stars  = ('***' if pv < 0.01 else
               '**' if pv < 0.05 else
                '*' if pv < 0.10 else '')
    print(f"  {name:<22} {dm:>9.3f} {pv:>9.4f}  {stars}")
print('─'*70)
print("  *** p<0.01  ** p<0.05  * p<0.10  (Harvey et al. 1997, df = n–1)")
print(f"  Note: n = {n_test} test obs — DM test has limited power in small samples.")

In [ ]:
fig = plt.figure(figsize=(15, 14))
fig.suptitle('Inflation Forecast — Model Comparison', fontsize=13, fontweight='bold', y=1.01)

# ── Panel 1: RMSE bar chart ───────────────────────────────────────────────
ax1 = fig.add_subplot(3, 1, 1)
colors = ['tomato' if m == 'FFNN (DNN)' else 'steelblue' for m in results_df['Model']]
bars   = ax1.barh(results_df['Model'], results_df['RMSE'],
                  color=colors, edgecolor='white', height=0.6)
for bar, val in zip(bars, results_df['RMSE']):
    ax1.text(val + 1e-4, bar.get_y() + bar.get_height() / 2,
             f'{val:.4f}', va='center', fontsize=9)
ax1.invert_yaxis()
ax1.set_xlabel('RMSE (percentage points)')
ax1.set_title('Model RMSE — Test Set  (↓ better; red = DNN)', fontweight='bold')
ax1.grid(True, axis='x', alpha=0.3)

# ── Panel 2: Forecast lines — Actual vs FFNN vs best traditional ──────────
best_trad_name = results_df[results_df['Model'] != 'FFNN (DNN)'].iloc[0]['Model']
best_trad_fcst = model_preds_dict[best_trad_name]

ax2 = fig.add_subplot(3, 1, 2)
ax2.plot(test_idx, y_test_raw,    label='Actual',          color='black',     lw=2.2)
ax2.plot(test_idx, y_pred,        label='FFNN (DNN)',      color='tomato',    lw=1.8, ls='--')
ax2.plot(test_idx, best_trad_fcst, label=best_trad_name,  color='steelblue', lw=1.8, ls='-.')
ax2.plot(test_idx, rw_preds,      label='Random Walk',     color='grey',      lw=1.2, ls=':')
ax2.set_title(f'Actual vs FFNN vs {best_trad_name} vs Random Walk', fontweight='bold')
ax2.set_ylabel('Inflation (%)')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.25)

# ── Panel 3: TVP-AR-SV — time-varying prediction intervals ───────────────
ci = 1.96 * np.sqrt(tvpar_sv_sig2)
ax3 = fig.add_subplot(3, 1, 3)
ax3.plot(test_idx, y_test_raw,     label='Actual',          color='black',      lw=2.0)
ax3.plot(test_idx, tvpar_sv_preds, label='TVP-AR-SV mean',  color='darkorange', lw=1.8, ls='--')
ax3.fill_between(test_idx,
                 tvpar_sv_preds - ci,
                 tvpar_sv_preds + ci,
                 alpha=0.25, color='darkorange', label='±1.96σ (time-varying)')
ax3.set_title('TVP-AR-SV — SV Value-Add: Time-Varying Prediction Intervals', fontweight='bold')
ax3.set_ylabel('Inflation (%)')
ax3.set_xlabel('Date')
ax3.legend(fontsize=9)
ax3.grid(True, alpha=0.25)

plt.tight_layout()
plt.savefig('lecture14_demo_model_comparison.png', bbox_inches='tight')
plt.show()
print("Saved: lecture14_demo_model_comparison.png")

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════╗
║  LECTURE 14 DEMO — UPDATED SUMMARY                                   ║
╠══════════════════════════════════════════════════════════════════════╣
║  Sections 1-10 : Feedforward Neural Network (FFNN)                   ║
║  Section 11    : Model comparison — DNN vs traditional econometrics  ║
╠══════════════════════════════════════════════════════════════════════╣
║  Models benchmarked (same 20% hold-out test set):                    ║
║    Univariate  : Random Walk · AR(p) · TVP-AR · TVP-AR-SV            ║
║    Multivariate: VAR(p) · TVP-VAR · TVP-VAR-SV                       ║
║    Deep Learning: FFNN (Sections 5-8)                                ║
╠══════════════════════════════════════════════════════════════════════╣
║  Statistical concepts demonstrated in Section 11:                    ║
║    1. Out-of-sample forecast horse race (same evaluation window)     ║
║    2. Diebold-Mariano test for equal predictive accuracy             ║
║    3. Kalman filter — TVP-AR with random-walk coefficients           ║
║    4. EWMA stochastic volatility approximation (TVP-AR-SV)           ║
║    5. Rolling-window VAR for time-varying multivariate models        ║
║    6. SV improves prediction intervals, not point forecasts          ║
╠══════════════════════════════════════════════════════════════════════╣
║  Saved outputs:                                                       ║
║    lecture14_demo_model_comparison.png                                ║
╚══════════════════════════════════════════════════════════════════════╝
""")